<a href="https://colab.research.google.com/github/Strojove-uceni/2024-final-julia-vendy/blob/main/Detekce_dopravnich_nehod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Automatická detekce dopravních nehod pomocí YOLOv8**
<p>

# Autoři: Vendula Rusá, Iuliia Shestopalova



# *Abstrakt*
 Tento projekt je zaměřen na automatickou detekci dopravních nehod na základě záznamů z pouličních kamer za využití modelů YOLOv8. Naší hlavní motivací bylo umožnit automatické upozornění policie nebo všech složek integrovaného záchranného systému, aby byl zkrácen čas do jejich příjezdu na místo nehody. Oproti současné situaci, kdy musí být v okolí přítomen svědek, který manuálně zavolá o pomoc, naše řešení eliminuje tuto závislost na lidské přítomnosti. Věříme, že tento systém má potenciál zachránit životy tím, že zrychlí reakci záchranných složek, což je klíčové zejména v případech, kde každá minuta hraje roli.

 Dataset využitý pro trénování modelů byl vytvořen pomocí Roboflow a zahrnoval obrazy s nehodami i bez nich. Po trénování několika modelů jsme vybrali nejlepší model na základě metrik a implementovali jsme GUI, které upozorňuje na detekci nehody.

# Metodologie
 Nejprve jsme vytvořili dataset kombinací veřejných dat a vlastní anotace. Použili jsme Roboflow pro snadnou anotaci a přípravu dat.

 Na obrázcích jsme chtěli anotovat výskyt nehody nebo vážné nehody. Kategorii "nehoda" pro nás tvoří situace, kdy je třeba zavolat policii. Označení "vazna nehoda" značí, že je nutný kromě policie ještě příjezd hasičů či záchranné služby. Původní dataset na Roboflow zahrnoval čtyři kategorie nehod. Kvůli nevyváženosti kategorií a jejich těžkému vyhodnocení jsme se rozhodly některé kategorie sloučit a doplnit obrázky bez nehod. Na obrázky rovněž byla použita augmentace cutout a noise neboli vyříznutí části obrázku a přídání šumu.

Pro detekci nehod jsme zvolili modely YOLO, jež jsou pokročilé modely pro detekci objektů známé svou rychlostí a přesností. YOLO mí velmi dobře rozpoznávat auta, což pro nás znamená, že náš model pro detekci nehod by nemuselo být potřeba trénovat dlouho.


Níže můžeme vidět grafické znázornění metrik oficiálních modelů na našich datech, tak rovněž přetrénovaných modelů na našich datech na 100 epochách.




In [1]:
%%capture
# Install Roboflow
!pip install roboflow

# Install Ultralytics (YOLOv8)
!pip install ultralytics

# Install YOLO dependencies
!pip install torch torchvision torchaudio
!pip install matplotlib
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt
import zipfile
import os

In [2]:
%%capture
# data from roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="c0AKlRxmACmG83mIunVG")
project = rf.workspace("su2-d7z1e").project("road-accident-victim-detection-bytzk")
version = project.version(2)
dataset = version.download("yolov8")

In [ ]:
# Načtení modelů z GitHubu a další soubory
!git clone https://github.com/Strojove-uceni/2024-final-julia-vendy.git

# Extrahování videí ze ZIP souboru
#zip_path = "su2projekt/videos.zip"
#unzip_dir = "su2projekt/videos"
#with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#    zip_ref.extractall(unzip_dir)

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np

def plot_metrics(models):

  # Test data path
  test_data_path = '/content/Road-accident-victim-detection-2/data.yaml'  # Assuming 'test' is defined in the .yaml file

  # Initialize metrics storage
  model_names = []
  metrics_data = {
      'Precision': [],
      'Recall': [],
      'mAP50': [],
      'mAP50-95': []
  }

  # Collect metrics for each model
  for mod in models:
      print(f"Validating model: {mod}")
      if mod.endswith('best.pt'):
        model_names.append(mod.replace('best.pt', ''))  # Remove the .pt extension from model name
        model = YOLO(mod)
      model_names.append(mod.replace('.pt', ''))  # Remove the .pt extension from model name
      model = YOLO(mod)

      # Validate the model on the test set
      metrics = model.val(data=test_data_path, split='test', imgsz=640)

      # Store metrics
      metrics_data['Precision'].append(metrics.box.p)
      metrics_data['Recall'].append(metrics.box.r)
      metrics_data['mAP50'].append(metrics.box.map50)
      metrics_data['mAP50-95'].append(metrics.box.map)


  # Extract class-specific metrics
  precision_nehoda = [prec[0] for prec in metrics_data['Precision']]
  recall_nehoda = [rec[0] for rec in metrics_data['Recall']]
  precision_vazna_nehoda = [prec[1] for prec in metrics_data['Precision']]
  recall_vazna_nehoda = [rec[1] for rec in metrics_data['Recall']]

  # Compute mean precision and recall
  mean_precision = [np.mean(prec) for prec in metrics_data['Precision']]
  mean_recall = [np.mean(rec) for rec in metrics_data['Recall']]


  # Plot 1: Precision and Recall for nehoda
  plt.figure(figsize=(12, 10))

  plt.subplot(2, 2, 1)  # First subplot for Precision and Recall - nehoda
  plt.plot( precision_nehoda, models, label='Precision - nehoda', marker='o')
  plt.plot(models, recall_nehoda, label='Recall - nehoda', marker='o')
  plt.title('Precision a Recall pro kategorii - nehoda')
  plt.ylabel('Modely')
  plt.xlabel('Metrické hodnoty')
  plt.legend()
  plt.grid()

  plt.subplot(2, 2, 2)  # Second subplot for Precision and Recall - vazna nehoda
  plt.plot(precision_vazna_nehoda, models, label='Precision - vazna nehoda', marker='o')
  plt.plot(models, recall_vazna_nehoda, label='Recall - vazna nehoda', marker='o')
  plt.title('Precision a Recall pro kategorii - vazna nehoda')
  plt.ylabel('Modely')
  plt.xlabel('Metrické hodnoty')
  plt.legend()
  plt.grid()

  plt.subplot(2, 1, 2)  # Third subplot for Aggregate Metrics
  plt.plot(mean_precision, models, label='Mean Precision', marker='o')
  plt.plot(models, mean_recall, label='Mean Recall', marker='o')
  plt.plot(models, metrics_data['mAP50'], label='mAP50', marker='o')
  plt.plot(models, metrics_data['mAP50-95'], label='mAP50-95', marker='o')
  plt.title('Průměrné metriky')
  plt.ylabel('Modely')
  plt.xlabel('Metrické hodnoty')
  plt.legend()
  plt.grid()

  plt.tight_layout()
  plt.show()



In [ ]:
# List of model paths
models = [
    'yolov8s.pt',
    'yolo11s.pt',
    'yolov8n_ac_best.pt',
    'yolov11s_ac_best.pt'
]

plot_metrics(models)

### Architektura modelu

Model byl modifikován na základě původní architektury YOLOv8s. Následující tabulka shrnuje klíčové komponenty modelu:

| **Vrstva**   | **Parametry**             |
|--------------|---------------------------|
| SPPF         | `[512, 512, 5]`          |
| Detect       | `[2, [128, 256, 512]]`   |
| C2f          | Více konfigurací, např. `[64, 64, 1, True]` |

### Shrnutí rozdílů oproti YOLOv8s

Hlavní rozdíly oproti oficiální implementaci YOLOv8s zahrnují:
- **SPPF vrstva**: Změněna velikost jádra na 5 místo 3, což zvyšuje schopnost modelu zachytit větší kontext.
- **Detect vrstva**: Přizpůsobena pro dvě třídy (2) oproti standardním 80 třídám v COCO datasetu.
- **C2f bloky**: Optimalizované konfigurace bloků s různým počtem kanálů a opakováním.
- **Parametry**: Celkový počet parametrů modelu je 11,136,374.

Modifikovaná architektura je navržena pro specifické potřeby projektu a nabízí vyšší efektivitu pro cílovou úlohu detekce s dvěma třídami. Tyto úpravy optimalizují model jak z hlediska výpočetních požadavků, tak z hlediska přesnosti pro specifický dataset.


Následně jsme experimentovali s různými konfiguracemi váhových parametrů modelu (váhy pro loss bounding boxů, klasifikace a objektovost). Byly definovány čtyři různé konfigurace:
 - Default_Weights: Základní váhy pro všechny ztráty.
 - Increased_Box_Loss: Zvýšená váha pro loss bounding boxů.
 - Increased_Cls_Loss: Zvýšená váha pro klasifikační loss.
 - Increased_Obj_Loss: Zvýšená váha pro loss objektovosti.


\begin{array}{|c|c|c|c|}
\hline
\textbf{Name} & \textbf{Box} & \textbf{Cls} & \textbf{Obj} \\\\
\hline
Default\_Weights & 0.05 & 0.5 & 1.0 \\\\
\hline
Increased\_Box\_Loss & 0.1 & 0.5 & 1.0 \\\\
\hline
Increased\_Cls\_Loss & 0.05 & 0.6 & 1.0 \\\\
\hline
Increased\_Obj\_Loss & 0.05 & 0.5 & 1.2 \\\\
\hline
\end{array}



Každá z těchto konfigurací byla trénována na 100 epochách pomocí YOLOv8s varianty, která je rychlejší a vhodná pro experimentování. Trénování probíhalo na GPU, pokud bylo dostupné, s optimalizovaným datovým tokem a parametry pro každou experimentální konfiguraci. Po vyhodnocení metrik přesnosti, preciznosti a rychlosti jsme vybrali nejlépe fungující konfiguraci.

Vybraný model jsme dále trénovali na 1000 epoch, abychom maximalizovali jeho přesnost. Model byl poté integrován do GUI rozhraní, které umožňuje zpracování dat z CCTV kamer v reálném čase a generování automatických upozornění.

Tento přístup kombinuje flexibilitu experimentování s váhovými parametry, robustnost moderních metod strojového učení a praktickou implementaci do reálného prostředí. Výsledky ukazují, že model je schopen přesně detekovat nehody, což poskytuje základ pro jeho nasazení v praktických aplikacích.

# *Výsledky*


První představu o našem modelu je možné si udělat na základě zobrazení několika obrázků z testovací části datasetu.

In [ ]:
# Funkce pro zobrazení detekce na obrázcích
import glob
from PIL import Image

def zobraz_detekce_na_obrazcich(model, image_dir, num_images):
    image_paths = glob.glob(f"{image_dir}/*.jpg")[:num_images]
    for img_path in image_paths:
        results = model.predict(source=img_path, save=True)
        img = Image.open(results[0].path)
        plt.imshow(img)
        plt.axis('off')
        plt.title("Detekovaná nehoda")
        plt.show()

# Ukázka detekce na prvních 5 obrázcích
image_dir = "su2projekt/test_images"
zobraz_detekce_na_obrazcich(model, image_dir, 5)

 Model dosáhl přesnosti X% na validačním datasetu a dokázal spolehlivě rozpoznat nehody i na testovacích obrázcích.
 Níže uvádíme grafy a tabulky s metrikami modelu. (Prostor pro vaše grafy/tabulky)


# *Ukázka použití v praxi*

In [ ]:

# Načtení GUI souboru
import sys
sys.path.append("su2projekt")
from gui_interface import start_gui

# Spuštění GUI
start_gui(model, video_dir=unzip_dir)


#Závěr:
 Tento projekt demonstruje možnost využití hlubokého učení pro automatizaci detekce dopravních nehod. Mezi limity patří omezený dataset a citlivost modelu na světelné podmínky.

# Reference:
 - https://universe.roboflow.com/dronemed/road-accident-victim-detection/dataset/4
 - https://www.kaggle.com/datasets/ckay16/accident-detection-from-cctv-footage